# White(1983) 공간 인접성 지수(SP, Spatial Proximity Index) — Python 구현

**논문:** White, M.J. (1983). "The Measurement of Spatial Segregation." *American Journal of Sociology*, 88(5), 1008–1018.

**관련 참고:** Massey, D.S. & Denton, N.A. (1988). "The Dimensions of Residential Segregation." *Social Forces*, 67, 281–315.

---

## 이 노트북의 구성

| 섹션 | 내용 |
|------|------|
| 1 | 라이브러리 설치 및 불러오기 |
| 2 | 거리 계산 (구역 간 + 구역 내) |
| 3 | 근접성 함수 $c_{ij} = \exp(-d_{ij})$ |
| 4 | 평균 근접도 $P_{xx}, P_{yy}, P_{xy}, P_{tt}$ |
| 5 | SP 지수 (논문 핵심 수식) |
| 6 | 항등식 검증 |
| 7 | 가상 도시 데이터 데모 |
| 8 | 결과 시각화 |

---

## 핵심 수식 요약

**거리:**
$$d_{ij} = \sqrt{(x_i-x_j)^2+(y_i-y_j)^2}, \qquad d_{ii} \approx 0.6\sqrt{A_i}$$

**근접성 함수:**
$$c_{ij} = \exp(-d_{ij})$$

**평균 근접도:**
$$P_{xx} = \frac{\sum_i\sum_j x_i x_j c_{ij}}{X^2}, \qquad P_{tt} = \frac{\sum_i\sum_j t_i t_j c_{ij}}{T^2}$$

**SP 지수:**
$$SP = \frac{X \cdot P_{xx} + Y \cdot P_{yy}}{T \cdot P_{tt}}$$

**해석:** SP > 1.0 = 군집성 존재, SP = 1.0 = 분리 없음, SP < 1.0 = 강한 혼합

---

## 섹션 1: 라이브러리 설치 및 불러오기

### 필요한 라이브러리

- **numpy:** 행렬 연산(외적, 행렬 곱, 지수함수 등) 처리
- **pandas:** 표 형태의 데이터 출력
- **matplotlib:** 시각화 (거리 히트맵, SP 비교 그래프)
- **scipy:** 유클리드 거리 행렬 자동 계산 (선택 사항)

### 설치 (터미널)
```
pip install numpy pandas matplotlib scipy
```

In [ ]:
# [목적]
#   - 이 셀을 가장 먼저 실행할 것
#   - 이후 모든 계산에서 사용할 라이브러리를 불러옴
#
# [라이브러리 별칭 설명]
#   - import numpy as np   : numpy를 'np'로 사용 (np.array, np.exp 등)
#   - import pandas as pd  : pandas를 'pd'로 사용 (pd.DataFrame 등)
#   - from scipy.spatial.distance import cdist : 쌍별 거리 계산 함수

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.spatial.distance import cdist

# [한글 폰트 설정]
#   - macOS: 'AppleGothic'
#   - Windows: 'Malgun Gothic'
#   - Linux: 'NanumGothic' (설치 필요)
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
%matplotlib inline

print("라이브러리 로드 완료")
print(f"  numpy 버전:      {np.__version__}")
print(f"  pandas 버전:     {pd.__version__}")
print(f"  matplotlib 버전: {matplotlib.__version__}")

---

## 섹션 2: 거리 계산

### 수식

**구역 간 거리:**
$$d_{ij} = \sqrt{(x_i-x_j)^2+(y_i-y_j)^2}$$

**같은 구역 내부 평균 거리 (White 1983 근사):**
$$d_{ii} \approx 0.6\sqrt{A_i}$$

### 핵심 가정

- 가정 1: 서로 다른 tract에 사는 사람들은 모두 해당 tract의 중심점(centroid)에 산다고 가정
- 가정 2: 같은 tract 내부 평균 거리는 tract 면적의 함수로 근사

In [ ]:
def compute_within_dist(area):
    """
    같은 tract 내부 평균 거리를 면적으로 근사.

    [수식]  d_ii ≈ 0.6 * sqrt(A_i)

    [입력 인수]
        - area : 각 tract의 면적 벡터 (1차원 배열, 단위 km²)

    [출력]
        - 각 tract의 내부 평균 거리 (1차원 배열, 단위 km)
    """
    # [단계 1] numpy 배열 변환
    area = np.array(area, dtype=float)

    # [단계 2] 핵심 계산
    #   - np.sqrt(area) : 배열의 각 원소에 제곱근 적용
    #   - 0.6 * ...     : White(1983)의 근사 계수 적용
    return 0.6 * np.sqrt(area)


def compute_dist_matrix(coords, area):
    """
    완전 거리 행렬 계산 (대각선 = 구역 내부 거리).

    [수식]
        d_ij = sqrt((x_i-x_j)^2 + (y_i-y_j)^2)  (i != j)
        d_ii = 0.6 * sqrt(A_i)                    (i == j)

    [입력 인수]
        - coords : tract 중심점 좌표 (n x 2 배열, 1열=x, 2열=y, 단위 km)
        - area   : 각 tract의 면적 (1차원 배열, 길이 n, 단위 km²)

    [출력]
        - n x n 거리 행렬 (대각선 = 내부 거리)
    """
    # [단계 1] numpy 배열 변환
    coords = np.array(coords, dtype=float)
    area   = np.array(area,   dtype=float)

    # [단계 2] 구역 간 거리 행렬 계산
    #   - cdist(coords, coords, metric='euclidean')
    #     scipy 함수로 모든 좌표 쌍 간의 유클리드 거리를 한번에 계산
    #     결과: n x n 행렬 (대각선 = 0, 자기 자신 거리)
    d_matrix = cdist(coords, coords, metric='euclidean')

    # [단계 3] 대각선을 within-tract 거리로 교체
    #   - np.diag_indices(n) : 대각선 원소의 인덱스를 반환
    #   - d_matrix[idx] = ... : 대각선 원소를 내부 거리로 덮어씀
    n = len(area)
    diag_idx = np.diag_indices(n)
    d_matrix[diag_idx] = compute_within_dist(area)

    return d_matrix


# --- 간단한 테스트 ---
# [테스트 데이터]
#   - 2개 tract: (0,0)과 (3,4) → 거리 = sqrt(9+16) = 5.0
test_coords = np.array([[0, 0], [3, 4]])
test_area   = np.array([1.0, 1.0])
test_d = compute_dist_matrix(test_coords, test_area)
print("[거리 행렬 테스트]")
print(f"  tract 간 거리 = {test_d[0,1]:.1f} km  (기댓값: 5.0)")
print(f"  within-tract = {test_d[0,0]:.4f} km  (기댓값: {0.6*np.sqrt(1.0):.4f})")

---

## 섹션 3: 근접성 함수 $c_{ij}$

### 수식

$$c_{ij} = f(d_{ij}) = \exp(-d_{ij})$$

### 해석

- 거리가 0일 때: $\exp(0) = 1$ (최대 근접)
- 거리가 증가할수록 0에 수렴 (거리감쇄)
- 대안: $c_{ij} = 1/d_{ij}$ (역거리 가중치)

In [ ]:
def compute_proximity(d_matrix, method='exp'):
    """
    거리 행렬을 근접성 행렬로 변환.

    [수식]
        exp 방식:     c_ij = exp(-d_ij)
        inverse 방식: c_ij = 1 / d_ij

    [입력 인수]
        - d_matrix : compute_dist_matrix()의 출력 (n x n 거리 행렬)
        - method   : 'exp' (기본값) 또는 'inverse'

    [출력]
        - n x n 근접성 행렬 (값이 클수록 가까움)
    """
    d_matrix = np.array(d_matrix, dtype=float)

    if method == 'exp':
        # [지수 근접성 함수]
        #   - np.exp(-d_matrix) : 배열의 각 원소에 exp(-x) 적용
        #   - d=0  → exp(0) = 1.0
        #   - d=1  → exp(-1) ≈ 0.368
        #   - d=5  → exp(-5) ≈ 0.007
        return np.exp(-d_matrix)

    elif method == 'inverse':
        # [역거리 가중치]
        #   - np.where(조건, 참일때값, 거짓일때값)
        #   - 거리가 0이면 매우 큰 값(1e6)으로 대체하여 0나누기 오류 방지
        return np.where(d_matrix > 0, 1.0 / d_matrix, 1e6)

    else:
        raise ValueError("method는 'exp' 또는 'inverse'만 지원합니다.")


# --- 근접성 함수 시각화 ---
# [목적]
#   - exp와 inverse 방식의 거리-근접성 관계를 그래프로 비교
distances = np.linspace(0.01, 8, 300)  # 0.01km ~ 8km, 300개 구간

fig, ax = plt.subplots(figsize=(9, 5))

# [exp 방식 곡선]
#   - np.exp(-distances) : 각 거리에 대해 exp(-d) 계산
ax.plot(distances, np.exp(-distances),
        color='steelblue', lw=2.5, label=r'$c_{ij} = \exp(-d_{ij})$ (기본값)')

# [inverse 방식 곡선]
#   - 1/distances : 각 거리의 역수
#   - ylim 제한으로 인해 d→0 근방의 급등 부분은 잘림
ax.plot(distances, 1/distances,
        color='darkorange', lw=2.5, linestyle='--',
        label=r'$c_{ij} = 1/d_{ij}$ (역거리)')

ax.set_xlim(0, 8)
ax.set_ylim(0, 2.5)
ax.set_xlabel('거리 $d_{ij}$ (km)', fontsize=12)
ax.set_ylabel('근접성 $c_{ij}$', fontsize=12)
ax.set_title('근접성 함수 비교: exp vs 역거리', fontsize=13)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.6, label='기준선 c=1.0')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---

## 섹션 4: 평균 근접도 계산

### 수식

$$P_{xx} = \frac{\sum_i\sum_j x_i x_j c_{ij}}{X^2}, \qquad P_{yy} = \frac{\sum_i\sum_j y_i y_j c_{ij}}{Y^2}$$

$$P_{xy} = \frac{\sum_i\sum_j x_i y_j c_{ij}}{X \cdot Y}, \qquad P_{tt} = \frac{\sum_i\sum_j t_i t_j c_{ij}}{T^2}$$

### 핵심 연산: 외적(outer product)

$$\sum_i\sum_j x_i x_j c_{ij} = \mathbf{x}^\top \cdot C \cdot \mathbf{x}$$

벡터 $\mathbf{x}$와 행렬 $C$의 이중 행렬곱으로 표현 가능하며, numpy에서는 `x @ C @ x` 또는 `np.outer(x,x) * C`로 계산한다.

In [ ]:
def compute_P_xx(x, cij):
    """
    집단 X 내 평균 근접도.

    [수식]  P_xx = sum_i sum_j (x_i * x_j * c_ij) / X^2

    [입력 인수]
        - x   : 각 tract의 집단 X 인구 수 (1차원 배열, 길이 n)
        - cij : 근접성 행렬 (n x n 배열)

    [출력]
        - 실수: 집단 X의 평균 근접도 (값이 클수록 X 집단이 밀집)
    """
    x = np.array(x, dtype=float)
    X = x.sum()
    if X == 0:
        raise ValueError("집단 X의 전체 인구가 0입니다.")

    # [핵심 계산]
    #   방법 1: np.outer(x, x) * cij → 원소별 곱 후 합산
    #     - np.outer(x, x) : 외적, 결과[i,j] = x[i] * x[j]
    #     - * cij           : 원소별 곱셈
    #     - .sum()          : 전체 합산 = Σ_i Σ_j x_i * x_j * c_ij
    #
    #   방법 2: x @ cij @ x (이중 행렬 곱)
    #     - 방법 1과 동일 결과, 더 빠름
    return float(x @ cij @ x) / X**2


def compute_P_yy(y, cij):
    """
    집단 Y 내 평균 근접도.

    [수식]  P_yy = sum_i sum_j (y_i * y_j * c_ij) / Y^2

    [입력 인수]
        - y   : 각 tract의 집단 Y 인구 수 (1차원 배열, 길이 n)
        - cij : 근접성 행렬 (n x n 배열)

    [출력]
        - 실수: 집단 Y의 평균 근접도
    """
    y = np.array(y, dtype=float)
    Y = y.sum()
    if Y == 0:
        raise ValueError("집단 Y의 전체 인구가 0입니다.")
    # [x @ cij @ x 설명]
    #   - x @ cij : (1 x n) @ (n x n) = (1 x n) 벡터
    #   - (x @ cij) @ x : (1 x n) @ (n x 1) = 스칼라
    #   - 결과 = Σ_i Σ_j y_i * c_ij * y_j
    return float(y @ cij @ y) / Y**2


def compute_P_xy(x, y, cij):
    """
    집단 X와 집단 Y 간 평균 근접도.

    [수식]  P_xy = sum_i sum_j (x_i * y_j * c_ij) / (X * Y)

    [입력 인수]
        - x   : 각 tract의 집단 X 인구 수 (1차원 배열)
        - y   : 각 tract의 집단 Y 인구 수 (1차원 배열)
        - cij : 근접성 행렬 (n x n 배열)

    [출력]
        - 실수: 집단 X-Y 간 평균 근접도 (항등식 검증에 사용)
    """
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    X, Y = x.sum(), y.sum()
    if X == 0 or Y == 0:
        raise ValueError("집단 X 또는 Y의 인구가 0입니다.")
    # [x @ cij @ y 설명]
    #   - x @ cij    : Σ_i x_i * c_ij (각 j에 대해 합산) → (n,) 벡터
    #   - (x@cij)@y  : Σ_j [...] * y_j = Σ_i Σ_j x_i * c_ij * y_j
    return float(x @ cij @ y) / (X * Y)


def compute_P_tt(t, cij):
    """
    전체 인구 평균 근접도.

    [수식]  P_tt = sum_i sum_j (t_i * t_j * c_ij) / T^2

    [입력 인수]
        - t   : 각 tract의 총 인구 수 (1차원 배열, t = x + y)
        - cij : 근접성 행렬 (n x n 배열)

    [출력]
        - 실수: 전체 인구 평균 근접도 (SP의 기준값)
    """
    t = np.array(t, dtype=float)
    T = t.sum()
    if T == 0:
        raise ValueError("전체 인구가 0입니다.")
    return float(t @ cij @ t) / T**2


print("평균 근접도 함수 정의 완료: compute_P_xx, compute_P_yy, compute_P_xy, compute_P_tt")

---

## 섹션 5: SP 지수 (논문 핵심 수식)

### 수식

$$SP = \frac{X \cdot P_{xx} + Y \cdot P_{yy}}{T \cdot P_{tt}}$$

### 동치 표현 (정규화된 형태)

$$SP = \frac{X\sum_i\sum_j p_{xi}p_{xj}f(d_{ij}) + Y\sum_i\sum_j p_{yi}p_{yj}f(d_{ij})}{(X+Y)\sum_i\sum_j p_i p_j f(d_{ij})}$$

단, $p_{xi} = x_i/X$, $p_{yi} = y_i/Y$, $p_i = t_i/T$

In [ ]:
def compute_SP(x, y, cij):
    """
    SP 지수 계산 (White 1983 핵심 수식).

    [수식]
        SP = (X * P_xx + Y * P_yy) / (T * P_tt)

    [입력 인수]
        - x   : 각 tract의 집단 X(소수집단) 인구 수 (1차원 배열, 길이 n)
        - y   : 각 tract의 집단 Y(다수집단) 인구 수 (1차원 배열, 길이 n)
        - cij : compute_proximity()의 출력 (n x n 근접성 행렬)

    [출력]
        - 실수: SP 지수 (1.0 기준)
          - SP > 1.0: 군집성 존재 (분리 심화)
          - SP = 1.0: 분리 없음 (무작위 분포)
          - SP < 1.0: 강한 혼합 (드문 경우)
    """
    # [단계 1] 배열 변환 및 기본 계산
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    t = x + y        # 각 tract 총 인구 (원소별 합산)

    X = x.sum()      # 도시 전체 X 인구
    Y = y.sum()      # 도시 전체 Y 인구
    T = X + Y        # 도시 전체 총 인구

    # [단계 2] 집단별 평균 근접도 계산
    P_xx = compute_P_xx(x, cij)   # 소수집단 X 내 근접도
    P_yy = compute_P_yy(y, cij)   # 다수집단 Y 내 근접도
    P_tt = compute_P_tt(t, cij)   # 전체 인구 근접도 (기준)

    # [단계 3] SP 지수 계산
    #   - 분자: 각 집단 내 근접도를 인구수로 가중 평균
    #           X*P_xx = X집단의 총 근접도 기여
    #           Y*P_yy = Y집단의 총 근접도 기여
    #   - 분모: 전체 인구 기준 근접도 (정규화 기준)
    numerator   = X * P_xx + Y * P_yy
    denominator = T * P_tt

    return numerator / denominator


def verify_identity(x, y, cij):
    """
    항등식 검증: T^2 * P_tt = X^2 * P_xx + 2*X*Y * P_xy + Y^2 * P_yy

    [의미]
        - 수학적으로 항상 성립해야 하는 항등식
        - 계산 오류 점검 도구

    [입력 인수]
        - x, y : 집단별 인구 벡터
        - cij  : 근접성 행렬

    [출력]
        - dict: 좌변값, 우변값, 차이값, 통과 여부
    """
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    t = x + y
    X, Y, T = x.sum(), y.sum(), t.sum()

    P_xx = compute_P_xx(x, cij)
    P_yy = compute_P_yy(y, cij)
    P_xy = compute_P_xy(x, y, cij)
    P_tt = compute_P_tt(t, cij)

    lhs = T**2 * P_tt
    rhs = X**2 * P_xx + 2*X*Y * P_xy + Y**2 * P_yy
    diff = abs(lhs - rhs)

    return {
        'lhs':    lhs,
        'rhs':    rhs,
        'diff':   diff,
        'passed': diff < 1e-6  # 허용 부동소수점 오차
    }


print("SP 지수 함수 정의 완료: compute_SP, verify_identity")

---

## 섹션 7: 가상 도시 데이터 데모 — 전체 계산 실행

### 가상 도시 설정

- 5개 tract으로 구성된 가상 도시
- 집단 X = 소수집단(Minority), 집단 Y = 다수집단(Majority)
- 북서쪽에 X 집중, 남동쪽에 Y 집중 → SP > 1.0 예상
- 좌표계: km 단위, 각 tract 면적 1 km²
- 근접성 함수: $c_{ij} = \exp(-d_{ij})$

In [ ]:
# ============================================================
# 데이터 설정
# ============================================================

# [인구 데이터]
#   - tract 1~2: 북서쪽, 소수집단(X) 우세
#   - tract 3  : 도심, 혼합 지역
#   - tract 4~5: 남동쪽, 다수집단(Y) 우세
tract_names = ['Tract1(NW)', 'Tract2(N)', 'Tract3(C)', 'Tract4(S)', 'Tract5(SE)']

x_pop = np.array([180, 150,  50,  30,  20], dtype=float)  # 소수집단 X
y_pop = np.array([ 20,  50, 100, 150, 180], dtype=float)  # 다수집단 Y
area  = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float) # 면적 (km²)

# [인구 데이터 출력]
t_pop = x_pop + y_pop
pop_df = pd.DataFrame({
    '구역':    tract_names,
    '소수(X)': x_pop.astype(int),
    '다수(Y)': y_pop.astype(int),
    '총인구':  t_pop.astype(int),
    'X비율':   (x_pop / t_pop).round(3)
})
print("[인구 데이터]")
print(pop_df.to_string(index=False))
print()

# [좌표 데이터]
#   - (1,5): 북서쪽 (Tract1)
#   - (5,1): 남동쪽 (Tract5)
#   - (3,3): 도심 (Tract3)
coords = np.array([
    [1, 5],  # Tract1(NW)
    [2, 4],  # Tract2(N)
    [3, 3],  # Tract3(C)
    [4, 2],  # Tract4(S)
    [5, 1],  # Tract5(SE)
], dtype=float)

coord_df = pd.DataFrame(coords, columns=['x(km)', 'y(km)'], index=tract_names)
print("[중심점 좌표]")
print(coord_df.to_string())
print()

In [ ]:
# ============================================================
# 계산 실행
# ============================================================

# --- 계산 1: 거리 행렬 ---
print("=" * 55)
print("[계산 1] 거리 행렬 (km)")
print("=" * 55)

d_matrix = compute_dist_matrix(coords, area)
d_df = pd.DataFrame(d_matrix.round(3), index=tract_names, columns=tract_names)
print(f"  대각선(within-tract) = 0.6*sqrt(1.0) = {0.6*np.sqrt(1.0):.4f} km")
print(d_df.to_string())
print()

# --- 계산 2: 근접성 행렬 ---
print("=" * 55)
print("[계산 2] 근접성 행렬 c_ij = exp(-d_ij)")
print("=" * 55)

cij = compute_proximity(d_matrix, method='exp')
cij_df = pd.DataFrame(cij.round(4), index=tract_names, columns=tract_names)
print(cij_df.to_string())
print()

# --- 계산 3: 평균 근접도 ---
print("=" * 55)
print("[계산 3] 집단별 평균 근접도")
print("=" * 55)

P_xx = compute_P_xx(x_pop, cij)
P_yy = compute_P_yy(y_pop, cij)
P_xy = compute_P_xy(x_pop, y_pop, cij)
P_tt = compute_P_tt(t_pop, cij)

print(f"  P_xx (소수집단 X 내 근접도)   = {P_xx:.6f}")
print(f"  P_yy (다수집단 Y 내 근접도)   = {P_yy:.6f}")
print(f"  P_xy (X-Y 집단 간 근접도)     = {P_xy:.6f}")
print(f"  P_tt (전체 인구 근접도)        = {P_tt:.6f}")
print()

# --- 계산 4: SP 지수 ---
print("=" * 55)
print("[계산 4] SP 지수")
print("=" * 55)

SP = compute_SP(x_pop, y_pop, cij)
print(f"  SP = {SP:.4f}")
print()

# --- 계산 5: 항등식 검증 ---
print("=" * 55)
print("[계산 5] 항등식 검증")
print("  T² × P_tt = X² × P_xx + 2XY × P_xy + Y² × P_yy")
print("=" * 55)

chk = verify_identity(x_pop, y_pop, cij)
print(f"  좌변 T² × P_tt           = {chk['lhs']:.6f}")
print(f"  우변 X²×P_xx+2XY×P_xy   = {chk['rhs']:.6f}")
print(f"  차이 |LHS - RHS|         = {chk['diff']:.2e}")
print(f"  검증 통과: {'YES' if chk['passed'] else 'NO'}")
print()

# --- 추가: inverse 방식 비교 ---
cij_inv = compute_proximity(d_matrix, method='inverse')
SP_inv  = compute_SP(x_pop, y_pop, cij_inv)
print("=" * 55)
print("[추가] 근접성 함수별 SP 비교")
print("=" * 55)
print(f"  SP (exp 방식):     {SP:.4f}")
print(f"  SP (inverse 방식): {SP_inv:.4f}")

In [ ]:
# ============================================================
# 최종 결과 요약
# ============================================================
X_total = x_pop.sum()
Y_total = y_pop.sum()
T_total = X_total + Y_total

print("=" * 60)
print("  White(1983) SP 지수 — 최종 결과 요약")
print("=" * 60)
print(f"  소수집단(X): {int(X_total)}명 ({100*X_total/T_total:.1f}%)")
print(f"  다수집단(Y): {int(Y_total)}명 ({100*Y_total/T_total:.1f}%)")
print(f"  도시 전체:   {int(T_total)}명")
print()

summary = pd.DataFrame({
    '지표':  ['P_xx', 'P_yy', 'P_tt', 'SP'],
    '값':    [round(P_xx, 6), round(P_yy, 6), round(P_tt, 6), round(SP, 4)],
    '의미':  ['소수집단 X 내 근접도', '다수집단 Y 내 근접도',
              '전체 인구 근접도(기준)', 'SP 지수(1.0 기준)']
})
print(summary.to_string(index=False))
print()
print("[해석]")
if SP > 1.0:
    print(f"  SP = {SP:.4f} > 1.0: 각 집단이 자기 집단끼리 더 가깝게 군집")
    print("    -> 공간적 군집성(분리) 존재. 소수집단이 특정 지역에 집중됨.")
elif SP < 1.0:
    print(f"  SP = {SP:.4f} < 1.0: 이질 집단이 오히려 서로 더 가깝게 섞임")
    print("    -> 강한 혼합 상태.")
else:
    print(f"  SP = {SP:.4f} = 1.0: 차별적인 군집성 없음.")

---

## 섹션 8: 결과 시각화

In [ ]:
# ============================================================
# 그림 1: 가상 도시 공간 분포 지도
# ============================================================
# [목적]
#   - 각 tract의 좌표와 인구 구성을 시각적으로 표현
#   - 원의 크기 = 총 인구, 색깔 = X 집단 비율

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 왼쪽: 공간 분포 ---
x_ratio = x_pop / t_pop  # 각 tract의 X 집단 비율

# [scatter 인수 설명]
#   - coords[:,0] : 좌표 배열의 첫번째 열 (x 좌표)
#   - coords[:,1] : 좌표 배열의 두번째 열 (y 좌표)
#   - s=t_pop*0.8 : 원의 면적 = 총 인구 x 0.8 (크기 비례)
#   - c=x_ratio   : 색깔 = X 집단 비율 (높을수록 진한 색)
#   - cmap='RdBu_r': 빨간(X 집단 높음) ~ 파란(Y 집단 높음) 색상
sc = axes[0].scatter(coords[:,0], coords[:,1],
                     s=t_pop*0.8, c=x_ratio,
                     cmap='RdBu_r', vmin=0, vmax=1,
                     edgecolors='black', linewidths=0.8, alpha=0.85)

# [tract 이름 표시]
for i, name in enumerate(tract_names):
    axes[0].annotate(name, (coords[i,0], coords[i,1]),
                     textcoords='offset points', xytext=(8,5), fontsize=8)

plt.colorbar(sc, ax=axes[0], label='X 집단 비율')
axes[0].set_xlabel('x 좌표 (km)', fontsize=11)
axes[0].set_ylabel('y 좌표 (km)', fontsize=11)
axes[0].set_title('가상 도시 공간 분포\n(원 크기=총인구, 색=X비율)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- 중앙: 근접성 히트맵 ---
# [imshow 인수 설명]
#   - cmap='YlGn' : 노란(낮음) ~ 초록(높음) 색상
#   - aspect='auto' : 셀 비율을 그림 크기에 맞게 자동 조정
im = axes[1].imshow(cij, cmap='YlGn', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], label='근접성 c_ij')
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))
axes[1].set_xticklabels(tract_names, rotation=20, fontsize=8)
axes[1].set_yticklabels(tract_names, fontsize=8)
axes[1].set_title('근접성 행렬 $c_{ij}=\\exp(-d_{ij})$', fontsize=11)

# [수치 표시]
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, f'{cij[i,j]:.3f}',
                     ha='center', va='center', fontsize=8,
                     color='black' if cij[i,j] < 0.6 else 'white')

# --- 오른쪽: 지수 비교 막대 ---
index_labels = ['$P_{xx}$\n(소수X 내)', '$P_{yy}$\n(다수Y 내)', '$P_{tt}$\n(전체)', 'SP']
index_vals   = [P_xx, P_yy, P_tt, SP]
colors_bar   = ['firebrick', 'steelblue', 'gray', 'darkorange']

bars = axes[2].bar(index_labels, index_vals, color=colors_bar, alpha=0.8, width=0.55)
axes[2].axhline(y=1.0, color='black', linestyle='--', lw=1.5, label='SP = 1.0 기준선')
axes[2].set_ylabel('지수 값', fontsize=11)
axes[2].set_title('White(1983) SP 지수 결과', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].grid(axis='y', alpha=0.4)

# [값 레이블]
for bar, val in zip(bars, index_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle(f'SP = {SP:.4f}  (> 1.0 → 군집성 존재)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 그림 2: 공간 배치에 따른 SP 변화 시뮬레이션
# ============================================================
# [목적]
#   - 인구 구성은 고정하고, 공간 배치만 바꿀 때 SP가 어떻게 변하는지 확인
#   - 시나리오 1: 완전 군집 (X가 한쪽에만)
#   - 시나리오 2: 완전 혼합 (X가 균등하게 분산)
#   - 시나리오 3: 원래 데이터 (점진적 그라디언트)

# [인구 총합 고정: X=430명, Y=500명]
total_x = x_pop.sum()  # 430
total_y = y_pop.sum()  # 500

# 시나리오별 인구 배분 정의
scenarios = {
    '완전 군집\n(X 모두 Tract1)': {
        'x': np.array([430,  0,  0,   0,   0], dtype=float),
        'y': np.array([100, 100, 100, 100, 100], dtype=float)
    },
    '강한 군집\n(원 데이터)': {
        'x': x_pop.copy(),
        'y': y_pop.copy()
    },
    '완전 혼합\n(X 균등 분산)': {
        'x': np.array([86, 86, 86, 86, 86], dtype=float),
        'y': np.array([100, 100, 100, 100, 100], dtype=float)
    }
}

# 각 시나리오에서 SP 계산
sp_results = {}
for name, data in scenarios.items():
    sp_val = compute_SP(data['x'], data['y'], cij)
    sp_results[name] = sp_val
    print(f"  {name.replace(chr(10), ' ')}: SP = {sp_val:.4f}")

print()

# 시각화
fig, ax = plt.subplots(figsize=(9, 5))

names = list(sp_results.keys())
vals  = list(sp_results.values())
bar_colors = ['firebrick', 'darkorange', 'steelblue']

bars = ax.bar(names, vals, color=bar_colors, alpha=0.8, width=0.5)
ax.axhline(y=1.0, color='black', linestyle='--', lw=1.5, label='SP = 1.0 기준선')
ax.set_ylabel('SP 지수 값', fontsize=12)
ax.set_title('공간 배치 시나리오별 SP 지수 비교\n(인구 총합은 고정, 공간 배치만 변경)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("[해석]")
print("  완전 군집 → SP 최대 (집단이 공간적으로 분리)")
print("  완전 혼합 → SP ≈ 1.0 (집단 간 공간 분포 차이 없음)")
print("  SP는 공간 배치에 민감하게 반응하는 군집성 지수임을 확인")